# Тур по gold-слою платформы

Готовый пример: как подключиться к данным, что где лежит и как построить график.
gold — витрины, готовые к анализу (read-only). Запусти ячейки по порядку (Shift+Enter).

**Каталог данных (OpenMetadata):** описания всех таблиц и колонок, происхождение (lineage) —
[meta.sk-vpn-2026.uk](https://meta.sk-vpn-2026.uk) (нужен VPN).
Прямая ссылка на таблицу удара: 
[fct_shot в каталоге](https://meta.sk-vpn-2026.uk/table/trino_iceberg.iceberg.gold.fct_shot).
FQN любой таблицы: `trino_iceberg.iceberg.gold.<имя>`.

## 1. Подключение к Trino
Логин/пароль и адрес уже прописаны в окружении контейнера (общий read-only аккаунт).
Писать в данные нельзя — только SELECT из `silver`/`gold`.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from trino.auth import BasicAuthentication

# SQLAlchemy-движок (pandas.read_sql хочет именно его — иначе UserWarning)
engine = create_engine(
    f"trino://{os.environ['TRINO_USER']}@{os.environ['TRINO_HOST']}:{os.environ['TRINO_PORT']}/iceberg",
    connect_args={
        "auth": BasicAuthentication(os.environ["TRINO_USER"], os.environ["TRINO_PASSWORD"]),
        "http_scheme": "https",
    },
)

def q(sql):
    """Выполнить SQL и вернуть DataFrame."""
    return pd.read_sql(sql, engine)

q("SELECT 1 AS ok")

## 2. Что лежит в gold
Две группы таблиц:
- **Словари (`dim_*`)** — справочники: команды, игроки, сезоны, лиги, судьи…
- **Факты (`fct_*`)** — события и метрики: удары, матчи, статистика игроков…

In [ ]:
tables = q("""
    SELECT table_name FROM iceberg.information_schema.tables
    WHERE table_schema = 'gold' ORDER BY 1
""")
dims = tables[tables.table_name.str.startswith('dim_')]
fcts = tables[tables.table_name.str.startswith('fct_')]
print(f"Словари ({len(dims)}):", ", ".join(dims.table_name))
print(f"\nФакты ({len(fcts)}):", ", ".join(fcts.table_name))

## 3. Словари — на примере `dim_team`
Словарь связывает технический `id` с человеческим именем. Пример — команды:

In [ ]:
q("SELECT team_id, team_name, short_name, country FROM iceberg.gold.dim_team ORDER BY team_name").head(10)

Остальные словари устроены так же — `id` + понятные атрибуты:

| Словарь | О чём |
|---|---|
| `dim_player` | игроки (имя, позиция, дата рождения) |
| `dim_competition` | турниры/лиги |
| `dim_season` | сезоны |
| `dim_match` | матчи (дата, команды, счёт) |
| `dim_manager` | тренеры |
| `dim_referee` | судьи |
| `dim_venue` | стадионы |
| `dim_player_attributes` | атрибуты игроков (FIFA-рейтинги) |

Колонки любой таблицы можно посмотреть так:

In [ ]:
q("""
    SELECT column_name, data_type FROM iceberg.information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'dim_player'
    ORDER BY ordinal_position
""")

## 4. Визуализация 1 — карта ударов (`fct_shot`)
`fct_shot` (~96 тыс. ударов) хранит координаты `x`,`y` (0..1 по полю), ожидаемые голы `xg`
и флаг гола. Построим карту ударов Ливерпуля за сезон 2024/25.

In [ ]:
shots = q("""
    SELECT s.x, s.y, s.xg, s.is_goal
    FROM iceberg.gold.fct_shot s
    JOIN iceberg.gold.dim_team t ON s.team_id = t.team_id
    WHERE t.team_name = 'Liverpool' AND s.season = '2425'
      AND s.x IS NOT NULL AND s.y IS NOT NULL
""")
print(f"ударов: {len(shots)}, из них голов: {int(shots.is_goal.sum())}")

fig, ax = plt.subplots(figsize=(10, 7))
goals = shots[shots.is_goal]
misses = shots[~shots.is_goal]
ax.scatter(misses.x, misses.y, s=misses.xg * 400 + 10, c="#888", alpha=0.5, label="промах")
ax.scatter(goals.x, goals.y, s=goals.xg * 400 + 10, c="#d00", alpha=0.8, label="гол")
ax.set_title("Liverpool 2024/25 — карта ударов (размер = xG)")
ax.set_xlabel("вдоль поля →"); ax.set_ylabel("поперёк поля")
ax.legend(); ax.set_xlim(0.5, 1.02); ax.set_ylim(-0.02, 1.02)
plt.show()

## 5. Визуализация 2 — топ команд по xG за сезон
Агрегация по всей таблице ударов + join со словарём команд.

In [ ]:
top = q("""
    SELECT t.team_name, sum(s.xg) AS total_xg, sum(cast(s.is_goal AS int)) AS goals
    FROM iceberg.gold.fct_shot s
    JOIN iceberg.gold.dim_team t ON s.team_id = t.team_id
    WHERE s.season = '2425'
    GROUP BY t.team_name ORDER BY total_xg DESC LIMIT 12
""")

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top.team_name[::-1], top.total_xg[::-1], color="#3a7")
ax.set_title("Топ-12 команд по суммарному xG, сезон 2024/25")
ax.set_xlabel("сумма xG")
plt.tight_layout(); plt.show()
top

## 6. Большая таблица — `fct_event` (~4.8 млн строк)
Все действия во всех матчах. Агрегацию делает Trino на сервере — в ноутбук приезжает
только маленький результат. Посмотрим, когда по ходу матча происходит больше всего событий.

In [ ]:
by_minute = q("""
    SELECT minute, count(*) AS events
    FROM iceberg.gold.fct_event
    WHERE minute BETWEEN 0 AND 90
    GROUP BY minute ORDER BY minute
""")

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(by_minute.minute, by_minute.events, color="#47a")
ax.set_title("Активность по минутам матча (fct_event, вся история)")
ax.set_xlabel("минута"); ax.set_ylabel("число событий")
plt.tight_layout(); plt.show()

## Дальше
- Меняй команду/сезон в запросах выше.
- Описания колонок — в каталоге [meta.sk-vpn-2026.uk](https://meta.sk-vpn-2026.uk).
- Хочешь, чтобы запросы шли под твоим именем (видно в истории Trino) — см.
  `docs/ANALYST_ONBOARDING.md`, вариант с `OAuth2Authentication`.
- Данные только для чтения: любой INSERT/UPDATE Trino отклонит.